# Environment

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from datasets import Dataset

from src.utils.path import DATA_PATH, GO_EMOTIONS_PATH

# Load GoEmotions

In [ ]:
model = BertForSequenceClassification.from_pretrained(GO_EMOTIONS_PATH)
tokenizer = BertTokenizer.from_pretrained(GO_EMOTIONS_PATH)

In [ ]:
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=256,
    top_k=None,
    return_all_scores=True,
    device=0
)

# Polarity Mapping

In [ ]:
polarity_mapping = {
    -1: {
        "aborrecimento", "constrangimento", "decepção", "desaprovação", "luto",
        "medo", "nervosismo", "nojo", "raiva", "remorso", "tristeza"
    },
    0: {
        "neutro", "curiosidade", "confusão", "percepção", "surpresa"
    },
    1: {
        "admiração", "alegria", "alívio", "aprovação", "amor", "desejo",
        "diversão", "entusiasmo", "gratidão", "orgulho", "otimismo", "zelo"
    }
}

In [ ]:
def map_emotions_to_polarity(emotions):
    polarities = []
    for emo in emotions:
        label = emo["label"].lower()
        score = emo["score"]
        if score >= 0.3:
            for polarity, emotion_set in polarity_mapping.items():
                if label in emotion_set:
                    polarities.append(polarity)
                    break

    total = sum(polarities)
    if total > 0: return 1
    if total < 0: return -1
    return 0

# Feature Engineering in Corpora

## Hate-BR

In [ ]:
hate = pd.read_csv(f"{DATA_PATH}/hate-br/processed/hate-br.csv")
hate_dataset = Dataset.from_pandas(hate[["text"]])

In [ ]:
hate_results = classifier(list(hate_dataset["text"]))
hate_polarities = [map_emotions_to_polarity(r) for r in hate_results]

In [ ]:
hate_polarities = pd.Series(hate_polarities, name="gep")

In [ ]:
hate_polarities.to_csv(f"{DATA_PATH}/hate-br/features/gep.csv", index=False)

## Ited-BR

In [ ]:
ited = pd.read_csv(f"{DATA_PATH}/ited-br/processed/ited-br.csv")
dataset = Dataset.from_pandas(ited[["text"]])

In [ ]:
ited_results = classifier(list(dataset["text"]))
ited_polarities = [map_emotions_to_polarity(r) for r in ited_results]

In [ ]:
ited_polarities = pd.Series(ited_polarities, name="gep")

In [ ]:
ited_polarities.to_csv(f"{DATA_PATH}/ited-br/features/gep.csv", index=False)

## Unified-Hate

In [ ]:
unified = pd.read_csv(f"{DATA_PATH}/unified-hate/processed/unified-hate.csv")
unified_dataset = Dataset.from_pandas(unified[["text"]])

In [ ]:
unified_results = classifier(list(dataset["text"]))
unified_polarities = [map_emotions_to_polarity(r) for r in unified_results]

In [ ]:
unified_polarities = pd.Series(unified_polarities, name="gep")

In [ ]:
unified_polarities.to_csv(f"{DATA_PATH}/unified-hate/features/gep.csv", index=False)